In [15]:
#Nombre Estudiante: Catalina Andrea Aguilera Letelier
#Generación: RTD-25-01-05-0019-1

#Consideraciones:
#Archivo: 07ventas_simuladas.csv (sep= ;)
#Librerías
from pyspark.sql import SparkSession
from datetime import datetime

#1 Cargar los datos ygenerar un RDD base
#Leer el archivo en Spark

#Configuración programática
spark = SparkSession.builder \
    .appName("VentasSimuladas") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.instances", "3") \
    .getOrCreate()

#Lectura de archivo spark
df_spark = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ";") \
    .csv("/content/07ventas_simuladas.csv")

#Crear un RDD base
sc = spark.sparkContext
rdd = sc.textFile("/content/07ventas_simuladas.csv")

#Aplicar una acción para visualizar las primeras líneas del dataset y verificar que los datos se han cargado correctamente
#Vista de un rdd es horizontal
print(rdd.take(5))

#Vista df_spark modo tabla vertical
df_spark.show(n=5, truncate=True, vertical=False)


['Sucursal;Producto;Cantidad;Precio_Unitario;Monto_Total;Fecha_Hora', 'Sucursal A;Pan;3;989.78;2969.34;2023-01-04 17:17:12', 'Sucursal A;Aceite;4;563.57;2254.28;2023-01-03 14:34:53', 'Sucursal B;Galletas;5;1339.04;6695.2;invalid-date', 'Sucursal B;Galletas;4;1180.5;4722.0;2023-01-05 2:34:24']
+----------+--------+--------+---------------+-----------+-------------------+
|  Sucursal|Producto|Cantidad|Precio_Unitario|Monto_Total|         Fecha_Hora|
+----------+--------+--------+---------------+-----------+-------------------+
|Sucursal A|     Pan|       3|         989.78|    2969.34|2023-01-04 17:17:12|
|Sucursal A|  Aceite|       4|         563.57|    2254.28|2023-01-03 14:34:53|
|Sucursal B|Galletas|       5|        1339.04|     6695.2|       invalid-date|
|Sucursal B|Galletas|       4|         1180.5|     4722.0| 2023-01-05 2:34:24|
|Sucursal D|     Pan|       3|        2194.99|    6584.97|2023-01-07 18:05:40|
+----------+--------+--------+---------------+-----------+----------------

In [35]:
#2 Aplicar transformaciones para limpiar y estructurar los datos
#Sobre el RDD base, deben realizar las siguientes transformaciones:
#Separar cada línea por comas y estructurar cada registro como una tupla
rdd_data = rdd.map(lambda linea: tuple(linea.split(";")))
print(rdd_data.take(4))

#Filtrar las filas incompletas o mal formateadas (por ejemplo, con campos vacíos o nulos)
print(f"Cantidad filas antes: {rdd_data.count()}") #Mostrar que esta todo bien
rdd_limpio = rdd_data.filter(lambda x: all(i is not None and i != '' and i != 'invalid-date' for i in x))
print(f"Cantidad filas después: {rdd_limpio.count()}")
print(rdd_limpio.take(4)) #Mostrar que esta todo bien

#Convertir los valores numéricos (como cantidad y monto) a su tipo correspondiente (int o float)
header = rdd_limpio.first()
rdd_sin_header = rdd_limpio.filter(lambda row: row != header)
rdd_modificado = rdd_sin_header.map(lambda x: (x[0], x[1], int(x[2]), float(x[3]), float(x[4]), x[5]))
print(rdd_modificado.take(3)) #Mostrar que esta todo bien

#Estandarizar el campo de fecha y hora, separándolo si es necesario
#Función para separar fecha y hora
def separar_fecha_hora(row):
    Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, ts_str = row

    # Convertir string a objeto datetime
    dt_obj = datetime.strptime(ts_str, '%Y-%m-%d %H:%M:%S')

    # Retornar (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, Fecha)
    return (Sucursal, Producto, Cantidad, Precio_Unitario, Monto_Total, dt_obj)

# Aplicar transformación
rdd_final = rdd_modificado.map(separar_fecha_hora)
print(rdd_final.take(3)) #Mostrar que esta todo bien

[('Sucursal', 'Producto', 'Cantidad', 'Precio_Unitario', 'Monto_Total', 'Fecha_Hora'), ('Sucursal A', 'Pan', '3', '989.78', '2969.34', '2023-01-04 17:17:12'), ('Sucursal A', 'Aceite', '4', '563.57', '2254.28', '2023-01-03 14:34:53'), ('Sucursal B', 'Galletas', '5', '1339.04', '6695.2', 'invalid-date')]
Cantidad filas antes: 1000
Cantidad filas después: 181
[('Sucursal', 'Producto', 'Cantidad', 'Precio_Unitario', 'Monto_Total', 'Fecha_Hora'), ('Sucursal A', 'Pan', '3', '989.78', '2969.34', '2023-01-04 17:17:12'), ('Sucursal A', 'Aceite', '4', '563.57', '2254.28', '2023-01-03 14:34:53'), ('Sucursal B', 'Galletas', '4', '1180.5', '4722.0', '2023-01-05 2:34:24')]
[('Sucursal A', 'Pan', 3, 989.78, 2969.34, '2023-01-04 17:17:12'), ('Sucursal A', 'Aceite', 4, 563.57, 2254.28, '2023-01-03 14:34:53'), ('Sucursal B', 'Galletas', 4, 1180.5, 4722.0, '2023-01-05 2:34:24')]
[('Sucursal A', 'Pan', 3, 989.78, 2969.34, datetime.datetime(2023, 1, 4, 17, 17, 12)), ('Sucursal A', 'Aceite', 4, 563.57, 2254

In [44]:
#3 Aplicar acciones de análisis y generar una síntesis del comportamiento de las ventas
#Utilizando acciones sobre el RDD limpio, generen al menos tres indicadores clave que serán útiles
#para construir un modelo de predicción futura. Por ejemplo:
#Total de ventas por sucursal
#(Sucursal, Monto_Total)
rdd_sucursal = rdd_final.map(lambda x: (x[0], float(x[4])))
rdd_sucursal_validado = rdd_sucursal.filter(lambda x: x[1] > 0) #Validar solo Montos positivos
total_ventas_sucursal = rdd_sucursal_validado.reduceByKey(lambda a, b: a + b) #Suma por sucursal
suma_ordenada = total_ventas_sucursal.sortBy(lambda x: x[1], ascending=False) #Ordenar total ventas de mayor a menor
print("Total de ventas por sucursal:", suma_ordenada.collect())

#Monto promedio por compra
rdd_montos = rdd_final.map(lambda x: x[4])
print(f"Monto promedio por compra: ${rdd_montos.mean():.2f}")

#Funcion Franjas horarias
def definir_franjas(valor):
    if 6 <= valor.hour < 12: return "Mañana"
    elif 12 <= valor.hour < 19: return "Tarde"
    elif 19 <= valor.hour <=23 : return "Noche"
    else: return "Madrugada"

#Mapear cada valor a su rango y luego sumar
conteo_rango = rdd_final.map(lambda x: (definir_franjas(x[5]), 1)).reduceByKey(lambda a, b: a + b)
print(conteo_rango.collect())



Total de ventas por sucursal: [('Sucursal C', 215554.92), ('Sucursal B', 211650.41000000003), ('Sucursal A', 196878.79), ('Sucursal D', 196034.5)]
Monto promedio por compra: $4556.21
[('Tarde', 58), ('Madrugada', 38), ('Noche', 35), ('Mañana', 49)]
